In [10]:
import numpy as np
import bz2

# Load sequences from a compressed BZ2 file
def load_sequences(filename):
    sequences = {}  # Create a dictionary to store gene sequences
    try:
        with bz2.open(filename, 'rt') as f:  # Open the file with read text mode
            for line in f:
                # Divide each line into gene ID and sequence parts
                parts = line.strip().split(' \\ ')
                if len(parts) == 2:
                    gene_id, sequence = parts[0].strip(), parts[1].strip()
                    sequences[gene_id] = sequence
                else:
                    print(f"Skipping malformed line: {line.strip()}")
    except Exception as e:
        print(f"Error reading or parsing file: {e}")
    print(f"Total sequences loaded: {len(sequences)}")
    return sequences

# Manually load the counts matrix from a text file
def load_counts_matrix(filepath):
    matrix = []
    with open(filepath) as f:
        for line in f:
            if '|' in line:
                base, numbers = line.strip().split('|')
                counts = [int(n) for n in numbers.strip().split()]
                matrix.append(counts)
    return np.array(matrix)

# Calculate PWM using counts matrix and pseudocounts
def compute_pwm(counts):
    pseudocount = 1
    counts_pseudocount = counts + pseudocount
    total_counts = counts_pseudocount.sum(axis=0)
    frequency_matrix = counts_pseudocount / total_counts
    background_frequency = 0.25
    return np.log2(frequency_matrix / background_frequency)

# Compute an adjusted frequency matrix using pseudocounts
def compute_adjusted_frequency(counts):
    pseudocount = 1
    return (counts + pseudocount) / (counts.sum(axis=0) + 4 * pseudocount)

# Search for the top binding sites using the PWM
def scan_sequences(pwm, sequences, top_n=30):
    base_mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    motif_length = pwm.shape[1]
    scores = []

    for gene_id, seq in sequences.items():
        if len(seq) < motif_length:
            print(f"Skipping {gene_id} due to short sequence length {len(seq)}")
            continue
        for i in range(len(seq) - motif_length + 1):
            subseq = seq[i:i+motif_length]
            score = sum(pwm[base_mapping.get(b, 0), j] for j, b in enumerate(subseq) if b in base_mapping)
            scores.append((gene_id, score, subseq, i))

    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_n]

# === MAIN EXECUTION ===

# File paths
bzipped_path = 'E_coli_K12_MG1655.400_50.bz2'
count_matrix_file = 'argR-counts-matrix.txt'

# Load data
sequences = load_sequences(bzipped_path)
counts = load_counts_matrix(count_matrix_file)

# Report sequence loading
if sequences:
    print("Sequences have been loaded successfully.")
else:
    print("No sequences were loaded.")

# Generate matrices
pwm = compute_pwm(counts)
adjusted_frequency = compute_adjusted_frequency(counts)

# Display matrices
print("\nPWM (Position Weight Matrix):")
print(pwm)

print("\nAdjusted Frequency Matrix F'(b, j) with Pseudocounts:")
print(adjusted_frequency)

# Find top motifs
top_binding_sites = scan_sequences(pwm, sequences)

# Show results
if top_binding_sites:
    print("\nTop 30 binding sites:")
    for site in top_binding_sites:
        print(f"Gene ID: {site[0]}, Score: {site[1]:.2f}, Position: {site[3]}, Subseq: {site[2]}")
else:
    print("No significant binding sites detected.")


Total sequences loaded: 4319
Sequences have been loaded successfully.

PWM (Position Weight Matrix):
[[ 0.21572869  0.74624341  1.50523531  0.36773178 -0.63226822 -1.36923381
   1.50523531  1.50523531 -0.95419631  0.50523531  0.21572869 -0.36923381
   0.04580369  1.74624341 -0.63226822 -1.36923381 -1.36923381  1.74624341]
 [ 0.04580369 -0.63226822 -1.95419631 -0.14684139 -1.36923381 -0.95419631
  -0.95419631 -0.95419631 -1.95419631 -2.95419631 -1.36923381 -2.95419631
   0.04580369 -2.95419631 -0.95419631 -0.95419631  1.68965988 -2.95419631]
 [-0.95419631 -1.36923381 -1.95419631  0.21572869 -1.36923381  1.50523531
  -1.36923381 -1.36923381 -2.95419631 -1.95419631 -2.95419631 -1.95419631
  -2.95419631 -1.95419631 -2.95419631  1.04580369 -2.95419631 -1.36923381]
 [ 0.36773178  0.36773178 -0.63226822 -0.63226822  1.36773178 -1.95419631
  -1.95419631 -1.95419631  1.63076619  1.13326653  1.21572869  1.50523531
   0.85315861 -1.95419631  1.43812111  0.04580369 -1.95419631 -2.95419631]]

Adjus